In [113]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    accuracy_score
)

In [114]:

# Load data
df = pd.read_csv("shop_smart_ecommerce.csv")

# Features and target
X = df.drop("Revenue", axis=1)
y = df["Revenue"].astype(int)

In [115]:
X.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True


In [116]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Revenue, dtype: int64

In [117]:
# Numerical and categorical columns
num_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

cat_features = X.select_dtypes(
    include=["object", "category"]
).columns

In [118]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=32,
    test_size=0.2, 
    stratify=y
)

In [119]:
X_train.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
9517,1,33.5,0,0.0,37,1702.916667,0.000000,0.009048,0.0,0.0,Nov,2,4,3,2,New_Visitor,False
6140,0,0.0,0,0.0,16,340.650000,0.060417,0.063690,0.0,0.0,Sep,3,2,2,3,Returning_Visitor,True
642,3,24.0,0,0.0,6,58.000000,0.000000,0.028571,0.0,0.0,Mar,2,2,8,2,Returning_Visitor,False
4631,0,0.0,0,0.0,2,39.000000,0.000000,0.100000,0.0,0.0,May,2,4,9,3,Returning_Visitor,False
3519,0,0.0,0,0.0,12,574.000000,0.016667,0.083333,0.0,0.0,May,2,2,1,13,Returning_Visitor,False


In [120]:
y_train.head()

9517    0
6140    0
642     0
4631    0
3519    0
Name: Revenue, dtype: int64

In [121]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

In [122]:
# Preprocessng Pipeline

# USE FOR COLUMN TRANSFORMER ::
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), num_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)


In [123]:
# Decision Tree
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=30,
    class_weight="balanced",
    random_state=42
)

In [124]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# Pipeline
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", dt)
])


In [125]:

# Parameter Grid
param_grid = {
    "model__max_depth": [4, 6, 8],
    "model__min_samples_leaf": [20, 30, 50]
}

In [126]:
# Grid Search
grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

# Train
grid.fit(X_train, y_train)

# Best parameters
print("Best F1:", grid.best_score_)
print("Best params:", grid.best_params_)

# Final prediction
y_pred = grid.predict(X_test)

# Evaluation
print("Test F1:", f1_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nAccuracy:", accuracy_score(y_test, y_pred))

Best F1: 0.647907854484464
Best params: {'model__max_depth': 4, 'model__min_samples_leaf': 20}
Test F1: 0.6586586586586587

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.86      0.91      2084
           1       0.53      0.86      0.66       382

    accuracy                           0.86      2466
   macro avg       0.75      0.86      0.79      2466
weighted avg       0.90      0.86      0.87      2466


Confusion Matrix:
 [[1796  288]
 [  53  329]]

Accuracy: 0.8617193836171938
